# Lab 1: Statistics Fundamentals

> **Course:** Data Science Foundations
> **Lesson:** Day 1 Lab, companion to `1-statistics-fundamentals.md`
> **Dataset:** `mpg` (seaborn). 398 cars from 1970 to 1982, with fuel efficiency, engine size, weight, and origin
> **Estimated time:** 90 minutes

## Lab Objectives

- Implement mean, median, variance, and standard deviation from scratch, and explain the `ddof` difference between NumPy and pandas
- Decide how to handle missing values and measure how that decision changes your statistics
- Build a descriptive summary table and use it to spot skewness and compare spread across scales
- Detect outliers with Tukey's fences and judge whether they are errors or real observations
- Measure relationships with covariance, Pearson, and **Spearman** correlation, and check them within subgroups
- Show by simulation that the spread of sample means follows the **standard error** formula $\sigma / \sqrt{n}$

**How to work through this lab**

- Cells marked `# TODO` are yours to complete. Replace each `None` or comment with working code.
- Run every **Checkpoint** cell. If it prints `passed`, move on. If it raises an error, fix your code first.
- Each **Question** needs a short written answer in the "Your answer" cell below it.
- The dataset is different from the lesson on purpose. Adapt the lesson code; don't paste it.
- Worked solutions are in the `solutions/` folder. Attempt every part yourself before you open them: the struggle is where the learning happens.

## Part 0: Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid")
rng = np.random.default_rng(42)

mpg = sns.load_dataset("mpg")
print(f"Shape: {mpg.shape}")
mpg.head()

## Part 1: Statistics by Hand

Libraries hide the arithmetic. Write your own versions of the core measures using only plain Python (`sum`, `len`, `sorted`, loops). **No NumPy or pandas in these functions.**

In [ ]:
def my_mean(values):
    # TODO: return the arithmetic mean
    return None


def my_median(values):
    # TODO: sort the values, then handle odd and even lengths
    return None


def my_variance(values, ddof=0):
    # TODO: average squared deviation from the mean.
    # ddof=0 divides by n (population); ddof=1 divides by n - 1 (sample, Bessel's correction)
    return None


def my_std(values, ddof=0):
    # TODO: square root of the variance
    return None

In [ ]:
# Checkpoint: run this cell. If it prints 'passed', move on.
# These are the worked examples from the lesson notes
assert my_mean([55, 60, 70, 80, 85]) == 70
assert my_median([55, 60, 70, 80, 85]) == 70
assert my_median([55, 60, 70, 80]) == 65
assert abs(my_variance([60, 70, 80]) - 66.667) < 0.001
assert abs(my_std([60, 70, 80]) - 8.165) < 0.001
assert abs(my_variance([60, 70, 80], ddof=1) - 100) < 1e-9
print("Part 1 checks passed")

### 1.2 Why do NumPy and pandas disagree?

Compute the standard deviation of the `mpg` column three ways and compare.

In [ ]:
mpg_values = mpg["mpg"]

std_numpy = None    # TODO: np.std(...) with its default settings
std_pandas = None   # TODO: mpg_values.std() with its default settings
std_mine = None     # TODO: your my_std, with the ddof that matches pandas

print(f"NumPy default : {std_numpy}")
print(f"pandas default: {std_pandas}")
print(f"my_std        : {std_mine}")

In [ ]:
# Checkpoint: run this cell. If it prints 'passed', move on.
assert abs(std_mine - std_pandas) < 1e-9
assert std_numpy < std_pandas
print("passed")

**Question 1.** Why do the NumPy and pandas values differ? Which one should you report for this dataset, and when would the difference actually matter?

**Your answer:**

*Write your answer here.*

## Part 2: Missing Values

The lesson dropped missing rows without much discussion. Here you will measure what that decision costs.

In [ ]:
mpg.isna().sum()

In [ ]:
# TODO 2.1: how many cars have a missing horsepower, and which cars are they?
n_missing = None
missing_cars = None   # DataFrame with columns name, model_year, horsepower for those rows

# TODO 2.2: build two versions of the horsepower column
hp_dropped = None     # missing values removed
hp_imputed = None     # missing values replaced by the median horsepower

# TODO 2.3: compare the mean, std and n of the two versions in one small DataFrame
comparison = None
comparison

**Question 2.** Median imputation lowered the standard deviation. Why does that always happen, and when would it become a real problem?

**Your answer:**

*Write your answer here.*

For the rest of the lab we drop the incomplete rows:

In [ ]:
cars = mpg.dropna().reset_index(drop=True)
numeric_cols = ["mpg", "displacement", "horsepower", "weight", "acceleration"]
print(f"Working with {len(cars)} complete cars")

## Part 3: Centre and Spread Summary

Build one table that answers "what is typical?" and "how much does it vary?" for every numeric column.

In [ ]:
# TODO 3.1: one row per column in numeric_cols, with these columns:
#   mean, median, mean_minus_median, std, IQR, CV_pct   (CV_pct = std / mean * 100)
summary = None
summary

In [ ]:
# Checkpoint: run this cell. If it prints 'passed', move on.
assert {"mean", "median", "mean_minus_median", "std", "IQR", "CV_pct"} <= set(summary.columns)
assert abs(summary.loc["weight", "CV_pct"] - cars["weight"].std() / cars["weight"].mean() * 100) < 1e-6
print("passed")

In [ ]:
# TODO 3.2: the mode of a categorical/discrete column
mode_cylinders = None   # most common number of cylinders
mode_origin = None      # most common origin
origin_share = None     # proportion of cars from each origin (value_counts with normalize=True)

print(mode_cylinders, mode_origin)
origin_share

**Question 3.**
(a) Which column has the largest spread in absolute terms, and which has the largest *relative* spread (CV)?
(b) Using `mean_minus_median`, which columns look right-skewed? Confirm one of them with a histogram in the cell below.

In [ ]:
# TODO: histogram (with KDE) of one column you think is right-skewed

**Your answer:**

*Write your answer here.*

## Part 4: Outliers with Tukey's Fences

Write reusable functions, then use them on several columns.

In [ ]:
def tukey_fences(series, k=1.5):
    # TODO: return (lower, upper) where lower = Q1 - k*IQR and upper = Q3 + k*IQR
    return None


def tukey_outliers(df, col, k=1.5):
    # TODO: return the rows of df whose value in `col` is outside the fences
    return None

In [ ]:
# Checkpoint: run this cell. If it prints 'passed', move on.
lo, hi = tukey_fences(pd.Series([1, 2, 3, 4, 100]))
assert (lo, hi) == (-1.0, 7.0)
assert len(tukey_outliers(pd.DataFrame({"x": [1, 2, 3, 4, 100]}), "x")) == 1
print("passed")

In [ ]:
for col in numeric_cols:
    lo, hi = tukey_fences(cars[col])
    print(f"{col:<13s} fences = [{lo:8.1f}, {hi:8.1f}]   outliers = {len(tukey_outliers(cars, col))}")

In [ ]:
# TODO 4.2: show the horsepower outliers (name, cylinders, origin, horsepower), sorted by horsepower
hp_outliers = None

# TODO 4.3: box plot of horsepower split by origin

**Question 4.** Are the horsepower outliers data errors? Should they be removed before analysis?

**Your answer:**

*Write your answer here.*

## Part 5: Relationships

### 5.1 Covariance by hand

The lesson's worked example (hours studied vs score) says the covariance is "positive" but never computes it. Compute it now with your Part 1 functions.

In [ ]:
hours = [2, 4, 6, 8]
scores = [50, 65, 80, 90]

cov_by_hand = None   # TODO: population covariance (divide by n)
r_by_hand = None     # TODO: cov / (std_x * std_y), using the matching ddof

print(f"Covariance: {cov_by_hand}")
print(f"Pearson r : {r_by_hand}")

In [ ]:
# Checkpoint: run this cell. If it prints 'passed', move on.
assert abs(cov_by_hand - 33.75) < 1e-9
assert abs(r_by_hand - np.corrcoef(hours, scores)[0, 1]) < 1e-9
print("passed. Note: np.cov(hours, scores) gives 45.0 because it divides by n - 1. r is unaffected.")

### 5.2 Correlation matrix

In [ ]:
# TODO: correlation matrix of numeric_cols, drawn as an annotated heatmap (vmin=-1, vmax=1)
corr = None

### 5.3 Pearson vs Spearman

Pearson's $r$ measures **linear** association. **Spearman's $\rho$** is Pearson's $r$ computed on the *ranks* of the values, so it measures any **monotonic** association (always increasing or always decreasing, even if curved). Run this demo first:

In [ ]:
x = np.linspace(0.1, 5, 200)
demo = pd.DataFrame({"x": x, "exp_x": np.exp(x), "parabola": (x - 2.5) ** 2})
print("y = exp(x):       Pearson =", round(demo["x"].corr(demo["exp_x"]), 3),
      " Spearman =", round(demo["x"].corr(demo["exp_x"], method="spearman"), 3))
print("y = (x - 2.5)^2:  Pearson =", round(demo["x"].corr(demo["parabola"]), 3),
      " Spearman =", round(demo["x"].corr(demo["parabola"], method="spearman"), 3))

In [ ]:
# TODO 5.3a: scatter plot of horsepower (x) vs mpg (y)

# TODO 5.3b: Pearson and Spearman correlation between horsepower and mpg
pearson_hp = None
spearman_hp = None
print(f"Pearson : {pearson_hp}")
print(f"Spearman: {spearman_hp}")

**Question 5.** Why is Spearman's value larger in magnitude than Pearson's for horsepower vs mpg? What does the parabola demo tell you that neither coefficient does?

**Your answer:**

*Write your answer here.*

## Part 6: Group Structure

On Day 1 the penguin species hid inside the aggregate numbers. Check whether `origin` plays the same role here.

In [ ]:
# TODO 6.1: count, mean, median and std of mpg for each origin
by_origin = None

# TODO 6.2: box plot of mpg by origin

# TODO 6.3: correlation between weight and mpg overall, and within each origin
overall_r = None
within_r = None   # Series indexed by origin

**Question 6.** A colleague says: *"Japanese cars get better mileage, so buying Japanese causes better fuel economy."* Use your results to explain what else could be going on.

**Your answer:**

*Write your answer here.*

## Part 7: Sampling Distributions and the Standard Error

The lesson showed that sample means vary *less* than individual values, but it did not say *how much* less. The answer is the **standard error of the mean**:

$$SE = \frac{\sigma}{\sqrt{n}}$$

You will confirm it by simulation, treating the 392 cars as the population.

In [ ]:
population = cars["mpg"].to_numpy()
sigma = population.std()          # population std (ddof=0) because this IS our population
sample_sizes = [5, 20, 80]
n_sims = 2000

# TODO 7.1: for each n, draw n_sims samples of size n (with replacement) and store the sample means
# Hint: rng.choice(population, size=(n_sims, n), replace=True).mean(axis=1)
sample_means = {}    # {n: array of n_sims means}

# TODO 7.2: table with index n and two columns: observed_std_of_means, theory_sigma_over_sqrt_n
se_table = None
se_table

In [ ]:
# Checkpoint: run this cell. If it prints 'passed', move on.
for n in sample_sizes:
    ratio = se_table.loc[n, "observed_std_of_means"] / se_table.loc[n, "theory_sigma_over_sqrt_n"]
    assert 0.9 < ratio < 1.1, f"n={n}: observed and theoretical SE differ too much"
print("passed")

In [ ]:
# TODO 7.3: a 1 x 4 figure: the population histogram, then the sampling distribution for each n.
# Use the same x-axis limits on all four panels so the narrowing is visible.

**Question 7.** By what factor does the standard error shrink when $n$ goes from 20 to 80? The population is right-skewed. What shape are the sampling distributions, and which theorem explains that?

**Your answer:**

*Write your answer here.*

## Part 8: Wrap-Up

Write 4 to 5 sentences for a non-technical manager summarising what you learned about fuel economy in this dataset. Include one number for centre, one for spread, one relationship, and one caveat about what this sample can and cannot tell us about cars in general.

**Your answer:**

*Write your answer here.*

> Submit your completed notebook to the Kanban board under your name by end of day.